In [1]:
"""
PIPELINE SARIMAX — VERSI HYPERPARAMETER TUNING
Sambungan dari prepro_modelling_iklim.ipynb

Perubahan dari versi baseline:
  1. Grid search diperluas: p,q ∈ {0,1,2,3}, P,Q ∈ {0,1,2}
  2. Kriteria seleksi: VAL MAE (bukan AIC training) → lebih realistis untuk forecasting
  3. ONI lag selection: uji kombinasi lag (lag1, lag1+2, lag1+2+3) per kabupaten
  4. d otomatis: ADF+KPSS per kabupaten, tidak dipaksa d=0 seragam
  5. Semua keputusan dilog ke CSV untuk reprodusibilitas
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import itertools
import pickle
from pathlib import Path
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import probplot, norm
from statsmodels.graphics.tsaplots import plot_acf

warnings.filterwarnings('ignore')

In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════
DIR_OUT  = Path(r"C:\Users\Keisha\Downloads\data sec\data")
FILE_CSV = DIR_OUT / "iklim_bulanan_kabupaten_3.csv"

SAMPEL = ['BADUNG', 'KOTA MALANG', 'JAYAPURA', 'KOTA SURABAYA']

TRAIN_END = '2019-12-01'
VAL_END   = '2022-12-01'

# ── Perubahan #1: Grid diperluas ─────────────────────────────────────
P_RANGE = [0, 1, 2, 3]   # baseline: [0,1,2]
Q_RANGE = [0, 1, 2, 3]   # baseline: [0,1,2]
SP_RANGE = [0, 1, 2]     # baseline: [0,1]
SQ_RANGE = [0, 1, 2]     # baseline: [0,1]

# ── Perubahan #3: Kandidat kombinasi ONI lag ─────────────────────────
# Akan diuji per kabupaten, dipilih yang terbaik di val
ONI_LAG_CANDIDATES = {
    'lag1'    : ['oni_lag1'],
    'lag1_2'  : ['oni_lag1', 'oni_lag2'],
    'lag1_2_3': ['oni_lag1', 'oni_lag2', 'oni_lag3'],
}

In [3]:
# ═══════════════════════════════════════════════════════════════════════
# HELPERS (sama seperti notebook asli, tidak diubah)
# ═══════════════════════════════════════════════════════════════════════
def stl_detrend(series, period=12, robust=True):
    s = series.dropna().copy()
    s.index = pd.to_datetime(s.index)
    s.index = s.index - pd.offsets.MonthBegin(0)
    if s.index.duplicated().any():
        s = s.groupby(s.index).mean()
    s = s.asfreq('MS')
    stl = STL(s, period=period, robust=robust)
    res = stl.fit()
    return {
        'detrended': pd.Series(res.seasonal + res.resid, index=s.index, name=series.name + '_detrended'),
        'trend'    : pd.Series(res.trend,                index=s.index, name=series.name + '_trend'),
        'seasonal' : pd.Series(res.seasonal,             index=s.index, name=series.name + '_seasonal'),
        'residual' : pd.Series(res.resid,                index=s.index, name=series.name + '_resid'),
        'stl_result': res,
    }

def build_oni_lags(series_oni, lags=(1, 2, 3)):
    df_lag = pd.DataFrame(index=series_oni.index)
    for lag in lags:
        df_lag[f'oni_lag{lag}'] = series_oni.shift(lag)
    return df_lag

def mae_fn(a, p):
    mask = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[mask] - p[mask]))

def rmse_fn(a, p):
    mask = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[mask] - p[mask])**2))

def mase_fn(a, p, s=12):
    mask = ~(np.isnan(a) | np.isnan(p))
    a, p = a[mask], p[mask]
    if len(a) <= s:
        return np.nan
    naive = np.mean(np.abs(a[s:] - a[:-s]))
    return np.mean(np.abs(a - p)) / naive if naive > 0 else np.nan

def cov95_fn(a, lo, hi):
    mask = ~(np.isnan(a) | np.isnan(lo) | np.isnan(hi))
    return ((a[mask] >= lo[mask]) & (a[mask] <= hi[mask])).mean() * 100

In [4]:
# ═══════════════════════════════════════════════════════════════════════
# PERUBAHAN #4: DETEKSI d OTOMATIS PER KABUPATEN
# Kombinasi ADF + KPSS → keputusan d lebih akurat
# ═══════════════════════════════════════════════════════════════════════
def detect_d(series, max_d=2):
    """
    Tentukan orde differencing d otomatis via ADF + KPSS.
    Logika:
      - Kalau keduanya stasioner (ADF tolak H0, KPSS terima H0) → d=0
      - Kalau tidak, differencing sekali dan cek lagi
    """
    s = series.dropna().copy()
    for d in range(max_d + 1):
        s_diff = s.diff(d).dropna() if d > 0 else s
        try:
            adf_p  = adfuller(s_diff, autolag='AIC')[1]
            kpss_p = kpss(s_diff, regression='c', nlags='auto')[1]
        except:
            return d
        # Stasioner: ADF tolak H0 (p<0.05) DAN KPSS tidak tolak H0 (p>0.05)
        if adf_p < 0.05 and kpss_p > 0.05:
            return d
    return max_d

In [5]:
# ═══════════════════════════════════════════════════════════════════════
# PERUBAHAN #2: GRID SEARCH DENGAN KRITERIA VAL MAE
# Bukan AIC training, tapi MAE di validation set
# Justifikasi: AIC mengoptimalkan fit training, bukan forecast accuracy
# Ref: Hyndman & Koehler 2006 — MASE sebagai metrik evaluasi forecast
# ═══════════════════════════════════════════════════════════════════════
def quick_val_forecast(endog_train, exog_train, endog_val, exog_val,
                       order, seasonal_order):
    """
    Fit pada train, forecast langsung ke val (bukan rolling, untuk kecepatan grid search).
    Return MAE di val dalam satuan asli.
    """
    try:
        model = SARIMAX(
            endog_train, exog=exog_train,
            order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False,
        )
        fit = model.fit(disp=False, maxiter=200)
        fc  = fit.forecast(steps=len(endog_val), exog=exog_val)
        return mae_fn(endog_val.values, fc.values), fit.aic
    except:
        return np.inf, np.inf


def grid_search_tuned(endog_train, exog_train, endog_val, exog_val,
                      d, seasonal_period=12):
    """
    Grid search SARIMAX dengan kriteria VAL MAE.
    p,q ∈ {0..3}, P,Q ∈ {0..2}, d dari detect_d, D=1
    """
    best_val_mae = np.inf
    best_order   = None
    best_seas    = None
    best_aic     = np.inf
    results      = []

    combos = list(itertools.product(P_RANGE, Q_RANGE, SP_RANGE, SQ_RANGE))
    print(f"      Grid: {len(combos)} kombinasi (p,q ∈ {{0..3}}, P,Q ∈ {{0..2}}), d={d}, D=1")

    for p, q, P, Q in combos:
        order     = (p, d, q)
        seas_ord  = (P, 1, Q, seasonal_period)
        val_mae, aic = quick_val_forecast(
            endog_train, exog_train, endog_val, exog_val, order, seas_ord
        )
        results.append({
            'order': order, 'seasonal_order': seas_ord,
            'val_mae': round(val_mae, 6), 'aic': round(aic, 2)
        })
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_order   = order
            best_seas    = seas_ord
            best_aic     = aic

    df_res = pd.DataFrame(results).sort_values('val_mae').reset_index(drop=True)
    return best_order, best_seas, best_val_mae, best_aic, df_res

In [6]:
# ═══════════════════════════════════════════════════════════════════════
# PERUBAHAN #3: SELEKSI ONI LAG TERBAIK PER KABUPATEN
# ═══════════════════════════════════════════════════════════════════════
def select_best_oni_lag(endog_train, exog_train_full, endog_val, exog_val_full,
                        order, seasonal_order):
    """
    Uji setiap kandidat ONI lag, pilih yang menghasilkan val MAE terkecil.
    """
    best_lag_name = None
    best_lag_mae  = np.inf

    for lag_name, lag_cols in ONI_LAG_CANDIDATES.items():
        # Pastikan semua kolom lag ada
        if not all(c in exog_train_full.columns for c in lag_cols):
            continue
        exog_tr = exog_train_full[lag_cols].dropna()
        exog_vl = exog_val_full[lag_cols].dropna()

        # Selaraskan index
        common_tr = endog_train.index.intersection(exog_tr.index)
        common_vl = endog_val.index.intersection(exog_vl.index)
        if len(common_tr) < 36 or len(common_vl) < 6:
            continue

        val_mae, _ = quick_val_forecast(
            endog_train.loc[common_tr], exog_tr.loc[common_tr],
            endog_val.loc[common_vl],   exog_vl.loc[common_vl],
            order, seasonal_order
        )
        if val_mae < best_lag_mae:
            best_lag_mae  = val_mae
            best_lag_name = lag_name

    return best_lag_name, ONI_LAG_CANDIDATES[best_lag_name] if best_lag_name else ['oni_lag1']

In [7]:
# ═══════════════════════════════════════════════════════════════════════
# ROLLING FORECAST (sama seperti notebook asli)
# ═══════════════════════════════════════════════════════════════════════
def rolling_forecast_sarimax(df_full, exog_cols, endog_col,
                              order, seasonal_order,
                              forecast_start, forecast_end,
                              refit_every=3):
    df_work = df_full.dropna(subset=[endog_col] + exog_cols).copy()
    preds, lowers, uppers, dates = [], [], [], []
    forecast_dates = pd.date_range(forecast_start, forecast_end, freq='MS')
    fit = None

    for i, fc_date in enumerate(forecast_dates):
        train_slice = df_work.loc[:fc_date - pd.DateOffset(months=1)]
        if len(train_slice) < 36:
            continue
        if i == 0 or (i % refit_every == 0):
            try:
                model = SARIMAX(
                    train_slice[endog_col],
                    exog=train_slice[exog_cols],
                    order=order, seasonal_order=seasonal_order,
                    enforce_stationarity=False, enforce_invertibility=False,
                )
                fit = model.fit(disp=False, maxiter=300)
            except:
                preds.append(np.nan); lowers.append(np.nan)
                uppers.append(np.nan); dates.append(fc_date)
                continue
        if fc_date not in df_work.index or fit is None:
            continue
        try:
            fc    = fit.get_forecast(steps=1, exog=df_work.loc[[fc_date], exog_cols])
            mean  = fc.predicted_mean.iloc[0]
            ci    = fc.conf_int(alpha=0.05)
            lower = ci.iloc[0, 0]; upper = ci.iloc[0, 1]
        except:
            mean, lower, upper = np.nan, np.nan, np.nan
        preds.append(mean); lowers.append(lower)
        uppers.append(upper); dates.append(fc_date)

    df_pred = pd.DataFrame({
        'tanggal': dates, 'predicted': preds,
        'lower95': lowers, 'upper95': uppers,
    }).set_index('tanggal')
    df_pred['actual'] = df_work.loc[df_pred.index, endog_col]
    return df_pred


def evaluate_forecast(df_pred, is_log=False, trend_series=None):
    a  = df_pred['actual'].values
    p  = df_pred['predicted'].values
    lo = df_pred['lower95'].values
    hi = df_pred['upper95'].values
    if is_log:
        a  = np.expm1(a)
        p  = np.expm1(np.clip(p, -10, 20))
        lo = np.expm1(np.clip(lo, -10, 20))
        hi = np.expm1(np.clip(hi, -10, 20))
    if trend_series is not None:
        tv = trend_series.reindex(df_pred.index).ffill().values
        a += tv; p += tv; lo += tv; hi += tv
    return {
        'MAE': round(mae_fn(a, p), 4),
        'RMSE': round(rmse_fn(a, p), 4),
        'MASE': round(mase_fn(a, p), 4),
        'Coverage95': round(cov95_fn(a, lo, hi), 1),
        'n': (~np.isnan(a)).sum(),
        '_actual': a, '_predicted': p,
        '_lower': lo, '_upper': hi,
        '_index': df_pred.index,
    }

In [8]:
# ═══════════════════════════════════════════════════════════════════════
# LOAD DATA + PREPROCESSING (sama seperti notebook asli)
# ═══════════════════════════════════════════════════════════════════════
print("=" * 70)
print("  SARIMAX PIPELINE — VERSI HYPERPARAMETER TUNING")
print("=" * 70)

df = pd.read_csv(FILE_CSV, parse_dates=['tanggal'])
df = df.sort_values(['kode_kab', 'tanggal']).reset_index(drop=True)
df_s = df[df['nama_kab'].isin(SAMPEL)].copy()
print(f"Data loaded: {len(df_s):,} baris, {df_s['nama_kab'].nunique()} kabupaten")

prepro_results = {}
split_info     = {}

for nama in SAMPEL:
    sub = (df_s[df_s['nama_kab'] == nama]
           .sort_values('tanggal').set_index('tanggal').copy())
    sub.index = pd.to_datetime(sub.index)
    sub.index = sub.index - pd.offsets.MonthBegin(0)
    sub = sub.asfreq('MS')

    stl_suhu    = stl_detrend(sub['suhu_rata_c'])
    hujan_log   = np.log1p(sub['curah_hujan_mm']).rename('curah_hujan_log')
    oni_lags    = build_oni_lags(sub['oni'], lags=(1, 2, 3))

    df_prepro = pd.concat([
        sub[['curah_hujan_mm', 'suhu_rata_c', 'oni', 'fase_enso']],
        stl_suhu['detrended'], stl_suhu['trend'],
        hujan_log, oni_lags,
    ], axis=1)

    prepro_results[nama] = {'df': df_prepro, 'stl_suhu': stl_suhu}

    train = df_prepro.loc[:TRAIN_END]
    val   = df_prepro.loc[pd.Timestamp(TRAIN_END) + pd.DateOffset(months=1):VAL_END]
    test  = df_prepro.loc[pd.Timestamp(VAL_END)   + pd.DateOffset(months=1):]
    split_info[nama] = {'train': train, 'val': val, 'test': test}

print("✅ Preprocessing & split selesai.\n")

  SARIMAX PIPELINE — VERSI HYPERPARAMETER TUNING
Data loaded: 960 baris, 4 kabupaten
✅ Preprocessing & split selesai.



In [9]:
# ═══════════════════════════════════════════════════════════════════════
# TUNING: GRID SEARCH + SELEKSI LAG ONI
# ═══════════════════════════════════════════════════════════════════════
print("=" * 70)
print("  GRID SEARCH TUNING (p,q ∈ {0..3}, P,Q ∈ {0..2}, kriteria: VAL MAE)")
print("  Ini akan memakan waktu lebih lama dari baseline...")
print("=" * 70)

best_orders  = {}
tuning_log   = []  # log semua keputusan untuk reprodusibilitas

for nama in SAMPEL:
    print(f"\n[{nama.upper()}]")
    train = split_info[nama]['train']
    val   = split_info[nama]['val']

    # ── CURAH HUJAN ─────────────────────────────────────────────────
    print("  ▶ Curah Hujan:")

    # Deteksi d otomatis
    d_hujan = detect_d(train['curah_hujan_log'])
    print(f"    d otomatis = {d_hujan}")

    # Awalnya pakai lag1+2 sebagai default untuk grid search awal
    exog_cols_default_h = ['oni_lag1', 'oni_lag2']
    endog_tr_h = train['curah_hujan_log'].dropna()
    exog_tr_h  = train[exog_cols_default_h].dropna()
    idx_h      = endog_tr_h.index.intersection(exog_tr_h.index)

    endog_vl_h = val['curah_hujan_log'].dropna()
    exog_vl_h  = val[exog_cols_default_h].dropna()
    idx_vh     = endog_vl_h.index.intersection(exog_vl_h.index)

    ord_h, seas_h, val_mae_h, aic_h, grid_h = grid_search_tuned(
        endog_tr_h.loc[idx_h], exog_tr_h.loc[idx_h],
        endog_vl_h.loc[idx_vh], exog_vl_h.loc[idx_vh],
        d=d_hujan
    )
    print(f"    Best order: {ord_h} x {seas_h}  VAL_MAE={val_mae_h:.4f}  AIC={aic_h:.1f}")

    # Seleksi lag ONI terbaik dengan order yang sudah dipilih
    print("    Seleksi lag ONI terbaik...")
    best_lag_name_h, best_lag_cols_h = select_best_oni_lag(
        endog_tr_h.loc[idx_h], train[['oni_lag1','oni_lag2','oni_lag3']].loc[idx_h],
        endog_vl_h.loc[idx_vh], val[['oni_lag1','oni_lag2','oni_lag3']].loc[idx_vh],
        ord_h, seas_h
    )
    print(f"    Best ONI lag: {best_lag_name_h} → {best_lag_cols_h}")

    # ── SUHU ─────────────────────────────────────────────────────────
    print("  ▶ Suhu Rata-rata:")

    d_suhu = detect_d(train['suhu_rata_c_detrended'])
    print(f"    d otomatis = {d_suhu}")

    exog_cols_default_s = ['oni_lag1']
    endog_tr_s = train['suhu_rata_c_detrended'].dropna()
    exog_tr_s  = train[exog_cols_default_s].dropna()
    idx_s      = endog_tr_s.index.intersection(exog_tr_s.index)

    endog_vl_s = val['suhu_rata_c_detrended'].dropna()
    exog_vl_s  = val[exog_cols_default_s].dropna()
    idx_vs     = endog_vl_s.index.intersection(exog_vl_s.index)

    ord_s, seas_s, val_mae_s, aic_s, grid_s = grid_search_tuned(
        endog_tr_s.loc[idx_s], exog_tr_s.loc[idx_s],
        endog_vl_s.loc[idx_vs], exog_vl_s.loc[idx_vs],
        d=d_suhu
    )
    print(f"    Best order: {ord_s} x {seas_s}  VAL_MAE={val_mae_s:.4f}  AIC={aic_s:.1f}")

    best_lag_name_s, best_lag_cols_s = select_best_oni_lag(
        endog_tr_s.loc[idx_s], train[['oni_lag1','oni_lag2','oni_lag3']].loc[idx_s],
        endog_vl_s.loc[idx_vs], val[['oni_lag1','oni_lag2','oni_lag3']].loc[idx_vs],
        ord_s, seas_s
    )
    print(f"    Best ONI lag: {best_lag_name_s} → {best_lag_cols_s}")

    best_orders[nama] = {
        'hujan': {
            'order': ord_h, 'seasonal': seas_h,
            'val_mae': val_mae_h, 'aic': aic_h,
            'd': d_hujan, 'oni_cols': best_lag_cols_h,
            'grid': grid_h
        },
        'suhu': {
            'order': ord_s, 'seasonal': seas_s,
            'val_mae': val_mae_s, 'aic': aic_s,
            'd': d_suhu, 'oni_cols': best_lag_cols_s,
            'grid': grid_s
        },
    }

    # Log keputusan
    tuning_log.append({
        'kabupaten': nama,
        'hujan_order': str(ord_h), 'hujan_seasonal': str(seas_h),
        'hujan_d': d_hujan, 'hujan_oni': str(best_lag_cols_h),
        'hujan_val_mae': val_mae_h, 'hujan_aic': aic_h,
        'suhu_order': str(ord_s), 'suhu_seasonal': str(seas_s),
        'suhu_d': d_suhu, 'suhu_oni': str(best_lag_cols_s),
        'suhu_val_mae': val_mae_s, 'suhu_aic': aic_s,
    })

# Simpan log tuning
pd.DataFrame(tuning_log).to_csv(DIR_OUT / 'tuning_log.csv', index=False)
print("\n✅ Grid search selesai. Log tersimpan: tuning_log.csv")

  GRID SEARCH TUNING (p,q ∈ {0..3}, P,Q ∈ {0..2}, kriteria: VAL MAE)
  Ini akan memakan waktu lebih lama dari baseline...

[BADUNG]
  ▶ Curah Hujan:
    d otomatis = 0
      Grid: 144 kombinasi (p,q ∈ {0..3}, P,Q ∈ {0..2}), d=0, D=1
    Best order: (0, 0, 0) x (2, 1, 2, 12)  VAL_MAE=0.2211  AIC=57.2
    Seleksi lag ONI terbaik...
    Best ONI lag: lag1_2 → ['oni_lag1', 'oni_lag2']
  ▶ Suhu Rata-rata:
    d otomatis = 0
      Grid: 144 kombinasi (p,q ∈ {0..3}, P,Q ∈ {0..2}), d=0, D=1
    Best order: (0, 0, 0) x (2, 1, 1, 12)  VAL_MAE=0.2184  AIC=60.1
    Best ONI lag: lag1_2_3 → ['oni_lag1', 'oni_lag2', 'oni_lag3']

[KOTA MALANG]
  ▶ Curah Hujan:
    d otomatis = 1
      Grid: 144 kombinasi (p,q ∈ {0..3}, P,Q ∈ {0..2}), d=1, D=1
    Best order: (1, 1, 0) x (2, 1, 1, 12)  VAL_MAE=0.2622  AIC=96.8
    Seleksi lag ONI terbaik...
    Best ONI lag: lag1 → ['oni_lag1']
  ▶ Suhu Rata-rata:
    d otomatis = 1
      Grid: 144 kombinasi (p,q ∈ {0..3}, P,Q ∈ {0..2}), d=1, D=1
    Best order: (3, 1

In [10]:
# ═══════════════════════════════════════════════════════════════════════
# ROLLING FORECAST DENGAN ORDER TERBAIK
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  ROLLING FORECAST (val + test)")
print("=" * 70)

forecast_results = {}
eval_results     = {}

for nama in SAMPEL:
    print(f"\n[{nama.upper()}]")
    df_prep   = prepro_results[nama]['df']
    stl_trend = prepro_results[nama]['stl_suhu']['trend']

    bh = best_orders[nama]['hujan']
    bs = best_orders[nama]['suhu']

    # ── Curah Hujan ─────────────────────────────────────────────────
    print(f"  → Rolling forecast curah hujan {bh['order']} x {bh['seasonal']}...")
    pred_h_val = rolling_forecast_sarimax(
        df_prep, bh['oni_cols'], 'curah_hujan_log',
        bh['order'], bh['seasonal'],
        split_info[nama]['val'].index.min(),
        split_info[nama]['val'].index.max(),
    )
    pred_h_test = rolling_forecast_sarimax(
        df_prep, bh['oni_cols'], 'curah_hujan_log',
        bh['order'], bh['seasonal'],
        split_info[nama]['test'].index.min(),
        split_info[nama]['test'].index.max(),
    )

    # ── Suhu ─────────────────────────────────────────────────────────
    print(f"  → Rolling forecast suhu {bs['order']} x {bs['seasonal']}...")
    pred_s_val = rolling_forecast_sarimax(
        df_prep, bs['oni_cols'], 'suhu_rata_c_detrended',
        bs['order'], bs['seasonal'],
        split_info[nama]['val'].index.min(),
        split_info[nama]['val'].index.max(),
    )
    pred_s_test = rolling_forecast_sarimax(
        df_prep, bs['oni_cols'], 'suhu_rata_c_detrended',
        bs['order'], bs['seasonal'],
        split_info[nama]['test'].index.min(),
        split_info[nama]['test'].index.max(),
    )

    forecast_results[nama] = {
        'hujan': {'val': pred_h_val, 'test': pred_h_test},
        'suhu' : {'val': pred_s_val, 'test': pred_s_test},
    }

    # ── Evaluasi ─────────────────────────────────────────────────────
    eval_results[nama] = {
        'hujan': {
            'val' : evaluate_forecast(pred_h_val,  is_log=True),
            'test': evaluate_forecast(pred_h_test, is_log=True),
        },
        'suhu': {
            'val' : evaluate_forecast(pred_s_val,  trend_series=stl_trend),
            'test': evaluate_forecast(pred_s_test, trend_series=stl_trend),
        },
    }
    print(f"  ✓ Selesai.")


  ROLLING FORECAST (val + test)

[BADUNG]
  → Rolling forecast curah hujan (0, 0, 0) x (2, 1, 2, 12)...
  → Rolling forecast suhu (0, 0, 0) x (2, 1, 1, 12)...
  ✓ Selesai.

[KOTA MALANG]
  → Rolling forecast curah hujan (1, 1, 0) x (2, 1, 1, 12)...
  → Rolling forecast suhu (3, 1, 3) x (2, 1, 2, 12)...
  ✓ Selesai.

[JAYAPURA]
  → Rolling forecast curah hujan (3, 0, 3) x (0, 1, 2, 12)...
  → Rolling forecast suhu (2, 0, 0) x (2, 1, 2, 12)...
  ✓ Selesai.

[KOTA SURABAYA]
  → Rolling forecast curah hujan (0, 1, 0) x (0, 1, 1, 12)...
  → Rolling forecast suhu (2, 1, 3) x (2, 1, 2, 12)...
  ✓ Selesai.


In [11]:
# ═══════════════════════════════════════════════════════════════════════
# PRINT EVALUASI
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 75)
print("  EVALUASI METRIK — SARIMAX TUNED")
print("  Dibandingkan dengan baseline (tanpa tuning) di kolom akhir")
print("=" * 75)

# Hasil baseline untuk perbandingan (dari notebook asli)
BASELINE = {
    'BADUNG'        : {'hujan': {'val': (2.660,0.871,94.4), 'test': (2.728,0.690,80.6)},
                       'suhu' : {'val': (0.294,0.835,86.1), 'test': (0.251,0.446,91.7)}},
    'KOTA MALANG'   : {'hujan': {'val': (2.563,0.731,88.9), 'test': (3.096,0.716,80.6)},
                       'suhu' : {'val': (0.347,0.819,88.9), 'test': (0.269,0.659,94.4)}},
    'JAYAPURA'      : {'hujan': {'val': (1.639,0.646,86.1), 'test': (1.654,0.562,88.9)},
                       'suhu' : {'val': (0.225,0.869,80.6), 'test': (0.201,0.620,94.4)}},
    'KOTA SURABAYA' : {'hujan': {'val': (2.786,0.749,86.1), 'test': (3.487,0.672,83.3)},
                       'suhu' : {'val': (0.323,0.849,88.9), 'test': (0.257,0.605,94.4)}},
}

summary_rows = []
for nama in SAMPEL:
    bh = best_orders[nama]['hujan']
    bs = best_orders[nama]['suhu']

    print(f"\n{'─'*75}")
    print(f"  {nama.upper()}")
    print(f"  Curah Hujan: SARIMAX{bh['order']}x{bh['seasonal']} | d={bh['d']} | ONI: {bh['oni_cols']}")
    print(f"  Suhu Rata  : SARIMAX{bs['order']}x{bs['seasonal']} | d={bs['d']} | ONI: {bs['oni_cols']}")
    print(f"\n  {'Variabel':14s} {'Split':5s} {'MAE':>8s} {'RMSE':>8s} {'MASE':>6s} {'Cov95':>7s} │ {'vs Baseline MAE':>16s}")
    print(f"  {'─'*70}")

    for var_label, var_key, unit in [('Curah Hujan', 'hujan', 'mm'), ('Suhu Rata', 'suhu', '°C')]:
        for split in ['val', 'test']:
            m   = eval_results[nama][var_key][split]
            bl  = BASELINE[nama][var_key][split]  # (MAE, MASE, Cov95)
            delta_mae = m['MAE'] - bl[0]
            arrow = '▼' if delta_mae < 0 else '▲'
            color_flag = ' ✅' if delta_mae < 0 else ' ⚠️'
            cov_flag = ' ⚠️ rendah' if m['Coverage95'] < 85 else ''
            mase_flag = ' ⚠️ >1' if m['MASE'] > 1 else ''

            print(f"  {var_label:14s} {split:5s} "
                  f"{m['MAE']:>7.3f}{unit} {m['RMSE']:>7.3f}{unit} "
                  f"{m['MASE']:>5.3f}{mase_flag} {m['Coverage95']:>6.1f}%{cov_flag} │ "
                  f"{arrow}{abs(delta_mae):.3f}{unit}{color_flag}")

            summary_rows.append({
                'kabupaten': nama, 'variabel': var_label, 'split': split,
                'MAE_tuned': m['MAE'], 'RMSE_tuned': m['RMSE'],
                'MASE_tuned': m['MASE'], 'Cov95_tuned': m['Coverage95'],
                'MAE_baseline': bl[0], 'MASE_baseline': bl[1], 'Cov95_baseline': bl[2],
                'delta_MAE': round(delta_mae, 4),
            })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(DIR_OUT / 'sarimax_tuned_eval_summary.csv', index=False)
print(f"\n{'─'*75}")
print("▼ = lebih baik dari baseline | ▲ = lebih buruk")
print("✅ Summary tersimpan: sarimax_tuned_eval_summary.csv")


  EVALUASI METRIK — SARIMAX TUNED
  Dibandingkan dengan baseline (tanpa tuning) di kolom akhir

───────────────────────────────────────────────────────────────────────────
  BADUNG
  Curah Hujan: SARIMAX(0, 0, 0)x(2, 1, 2, 12) | d=0 | ONI: ['oni_lag1', 'oni_lag2']
  Suhu Rata  : SARIMAX(0, 0, 0)x(2, 1, 1, 12) | d=0 | ONI: ['oni_lag1', 'oni_lag2', 'oni_lag3']

  Variabel       Split      MAE     RMSE   MASE   Cov95 │  vs Baseline MAE
  ──────────────────────────────────────────────────────────────────────
  Curah Hujan    val     2.714mm   3.267mm 0.889   88.9% │ ▲0.054mm ⚠️
  Curah Hujan    test    3.050mm   3.840mm 0.771   80.6% ⚠️ rendah │ ▲0.321mm ⚠️
  Suhu Rata      val     0.265°C   0.336°C 0.753   88.9% │ ▼0.029°C ✅
  Suhu Rata      test    0.220°C   0.305°C 0.391   91.7% │ ▼0.031°C ✅

───────────────────────────────────────────────────────────────────────────
  KOTA MALANG
  Curah Hujan: SARIMAX(1, 1, 0)x(2, 1, 1, 12) | d=1 | ONI: ['oni_lag1']
  Suhu Rata  : SARIMAX(3, 1, 3)x(2

In [12]:
# ═══════════════════════════════════════════════════════════════════════
# SIMPAN MODEL FINAL + FORECAST
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  SIMPAN MODEL FINAL (fit pada train+val)")
print("=" * 70)

for nama in SAMPEL:
    nama_slug = nama.lower().replace(' ', '_')
    df_prep   = prepro_results[nama]['df']
    bh = best_orders[nama]['hujan']
    bs = best_orders[nama]['suhu']

    for var_key, endog_col, oni_cols, order, seasonal in [
        ('hujan', 'curah_hujan_log', bh['oni_cols'], bh['order'], bh['seasonal']),
        ('suhu',  'suhu_rata_c_detrended', bs['oni_cols'], bs['order'], bs['seasonal']),
    ]:
        trainval = df_prep.loc[:VAL_END].dropna(subset=[endog_col] + oni_cols)
        model_f  = SARIMAX(
            trainval[endog_col], exog=trainval[oni_cols],
            order=order, seasonal_order=seasonal,
            enforce_stationarity=False, enforce_invertibility=False,
        ).fit(disp=False, maxiter=300)

        path_pkl = DIR_OUT / f'sarimax_tuned_{var_key}_{nama_slug}.pkl'
        with open(path_pkl, 'wb') as f:
            pickle.dump(model_f, f)

        # Export forecast CSV
        for split in ['val', 'test']:
            m = eval_results[nama][var_key][split]
            pd.DataFrame({
                'tanggal'  : m['_index'],
                'actual'   : m['_actual'],
                'predicted': m['_predicted'],
                'lower95'  : m['_lower'],
                'upper95'  : m['_upper'],
            }).to_csv(DIR_OUT / f'forecast_tuned_{var_key}_{nama_slug}_{split}.csv', index=False)

    print(f"  ✓ {nama} tersimpan.")

print("\n" + "=" * 70)
print("  PIPELINE SARIMAX TUNED SELESAI")
print(f"  Output di: {DIR_OUT}")
print("  File utama:")
print("    tuning_log.csv               → keputusan order & lag per kabupaten")
print("    sarimax_tuned_eval_summary.csv → perbandingan metrik vs baseline")
print("    sarimax_tuned_*.pkl           → model final (train+val)")
print("    forecast_tuned_*.csv          → hasil forecast val & test")
print("=" * 70)


  SIMPAN MODEL FINAL (fit pada train+val)
  ✓ BADUNG tersimpan.
  ✓ KOTA MALANG tersimpan.
  ✓ JAYAPURA tersimpan.
  ✓ KOTA SURABAYA tersimpan.

  PIPELINE SARIMAX TUNED SELESAI
  Output di: C:\Users\Keisha\Downloads\data sec\data
  File utama:
    tuning_log.csv               → keputusan order & lag per kabupaten
    sarimax_tuned_eval_summary.csv → perbandingan metrik vs baseline
    sarimax_tuned_*.pkl           → model final (train+val)
    forecast_tuned_*.csv          → hasil forecast val & test


In [13]:
# ── EXPORT TERPADU untuk perbandingan dengan model lain (TFT, dll) ────
all_preds = []

for nama in SAMPEL:
    stl_trend = prepro_results[nama]['stl_suhu']['trend']

    for var_key, unit, is_log in [('hujan', 'mm', True), ('suhu', '°C', False)]:
        for split in ['val', 'test']:
            m = eval_results[nama][var_key][split]

            df_exp = pd.DataFrame({
                'kabupaten' : nama,
                'variabel'  : var_key,
                'split'     : split,
                'tanggal'   : m['_index'],
                'actual'    : m['_actual'],
                'predicted' : m['_predicted'],
                'lower95'   : m['_lower'],
                'upper95'   : m['_upper'],
                'error_abs' : np.abs(m['_actual'] - m['_predicted']),
                'model'     : 'SARIMAX_tuned',
            })
            all_preds.append(df_exp)

df_all_preds = pd.concat(all_preds, ignore_index=True)
df_all_preds.to_csv(DIR_OUT / 'predictions_sarimax_tuned.csv', index=False)
print("✅ Prediksi terpadu tersimpan: predictions_sarimax_tuned.csv")

✅ Prediksi terpadu tersimpan: predictions_sarimax_tuned.csv
